# Step 6 — Final evaluation: every detector and baseline, same rules

**Purpose.** Score all methods with the one shared harness (`eval_core.py`, protocol §7), run the paired comparison, the sensitivity check (§10) and build the report's main table and figures. No GPU.

**Methods**
- **DistilBERT**, augmented (headline) and not augmented (ablation), seeds 42/43/44: mean ± sd across seeds; bootstrap CIs for seed 42.
- **Feature scorer** (logistic regression on retrieval score, z-score, centroid distance).
- **Baselines:** passage length, retrieval score (thresholds from `data/frozen/thresholds.json`), starts-with-query, ends-with-a-full-stop, simplified TrustRAG Stage-1 (the last three are yes/no rules: TPR/FPR at the fixed rule; TrustRAG gets no AUROC).

**Outputs** (Drive, `rag_trust/step6_eval/`): `detection_results.json`, `table1_detection.csv`, `paired_tests.json`, `sensitivity.json`, `table2_end_to_end.csv`, `latency.json`, `fig3_shortcuts.png/.pdf`, `fig4_end_to_end.png/.pdf`, and failure-analysis sheets (`failure_<method>.csv`) for hand categorisation.

## 1. Setup

In [ ]:
import os, sys, json, re, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/rag_trust"
else:
    DRIVE_ROOT = os.environ["RAG_TRUST_ROOT"]
OUT = f"{DRIVE_ROOT}/step6_eval"; os.makedirs(OUT, exist_ok=True)
N_BOOT, SEED, SEEDS = 2000, 42, [42, 43, 44]
FLAGGED = {("nq", "test442"), ("nq", "test53"), ("msmarco", "156641"), ("msmarco", "416509"), ("msmarco", "577564")}

In [ ]:
%%writefile eval_core.py
"""Shared evaluation for all detectors and baselines (eval_protocol.md section 7).

Every detector writes one score per row of data/passages.jsonl (higher = more likely
poisoned). This module turns rows + scores into the protocol's pairings, rates at frozen
thresholds, bootstrap CIs (resampling questions within each dataset) and the paired
AUROC difference. Pure NumPy, no GPU.
"""
import numpy as np

# dataset groups reported separately (section 4)
GROUPS = {"in_dist": ("nq", "hotpotqa"), "msmarco": ("msmarco",)}


# ---------------------------------------------------------------- metrics

def auroc(pos, neg):
    """AUROC of score(pos) > score(neg), ties counted 0.5 (Mann-Whitney)."""
    pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    if pos.size == 0 or neg.size == 0:
        return float("nan")
    allv = np.concatenate([pos, neg])
    _, inv, counts = np.unique(allv, return_inverse=True, return_counts=True)
    order = allv.argsort(kind="mergesort")
    ranks = np.empty(len(allv)); ranks[order] = np.arange(1, len(allv) + 1)
    ranks = (np.bincount(inv, weights=ranks) / counts)[inv]
    return float((ranks[:pos.size].sum() - pos.size * (pos.size + 1) / 2) / (pos.size * neg.size))


def threshold_at_fpr(neg_scores, fpr):
    """Threshold t such that flagging score > t gives FPR <= fpr on these negatives
    (section 7: set on val n=0 clean passages, then frozen)."""
    s = np.sort(np.asarray(neg_scores, float))[::-1]
    k = int(np.floor(fpr * len(s)))          # number of false positives allowed
    return float(s[k]) if k < len(s) else float(s[-1]) - 1e-12


def rate(scores, t):
    """Share of scores flagged (score > t): TPR on positives, FPR on negatives."""
    s = np.asarray(scores, float)
    return float((s > t).mean()) if s.size else float("nan")


# ---------------------------------------------------------------- pairings (section 7)

def _is(r, **kw):
    return all(r.get(k) == v for k, v in kw.items())


def pairing(rows, name):
    """Return (positive_rows, negative_rows) for one protocol pairing.
    Positives at n=1 use the five n=1 detection groups (headline condition)."""
    orig_pos = [r for r in rows if _is(r, group_variant="original", n_poison=1, label=1)]
    orig_neg = [r for r in rows if _is(r, group_variant="original", n_poison=0, label=0)]
    if name == "original":
        return orig_pos, orig_neg
    if name == "original_n5":
        return [r for r in rows if _is(r, group_variant="original", n_poison=5, label=1)], orig_neg
    if name == "prefix_poison_side":
        return [r for r in rows if _is(r, group_variant="prefix_removed", n_poison=1, variant="prefix_removed")], orig_neg
    if name == "prefix_poison_side_n5":   # all five poison prefix-removed (n=5 reported separately, protocol section 7)
        return [r for r in rows if _is(r, group_variant="prefix_removed", n_poison=5, variant="prefix_removed")], orig_neg
    if name == "prefix_clean_side":
        return orig_pos, [r for r in rows if _is(r, group_variant="clean_prefixed", variant="clean_prefixed")]
    if name == "llm_style":
        return orig_pos, [r for r in rows if _is(r, group_variant="rewritten_clean", variant="rewritten_clean")]
    if name == "shortcut_controlled":
        return ([r for r in rows if _is(r, group_variant="shortcut_controlled", n_poison=1, variant="prefix_removed")],
                [r for r in rows if _is(r, group_variant="shortcut_controlled", n_poison=0, variant="clean_truncated")])
    if name == "unseen_attack":
        pos = [r for r in rows if _is(r, group_variant="injection", variant="injection")]
        qs = {(r["dataset"], r["qid"]) for r in pos}
        return pos, [r for r in orig_neg if (r["dataset"], r["qid"]) in qs]
    if name == "collateral_n1":       # FPR only: clean passages inside attacked n=1 groups
        return [], [r for r in rows if _is(r, group_variant="original", n_poison=1, label=0)]
    raise ValueError(name)


PAIRINGS = ["original", "original_n5", "prefix_poison_side", "prefix_poison_side_n5", "prefix_clean_side", "llm_style",
            "shortcut_controlled", "unseen_attack", "collateral_n1"]


# ---------------------------------------------------------------- bootstrap by question

def _by_question(rs, score_key):
    out = {}
    for r in rs:
        out.setdefault((r["dataset"], r["qid"]), []).append(r[score_key])
    return out


def _resample_keys(keys, rng):
    """Resample questions within each dataset (so one dataset cannot dominate)."""
    by_ds = {}
    for k in keys:
        by_ds.setdefault(k[0], []).append(k)
    out = []
    for ds in sorted(by_ds):
        ks = by_ds[ds]
        out += [ks[i] for i in rng.integers(0, len(ks), len(ks))]
    return out


def evaluate(rows, score_key, thresholds, n_boot=2000, seed=42, pairings=PAIRINGS, groups=GROUPS):
    """Detection metrics for one score column. thresholds: {"fpr5": t5, "fpr1": t1}.
    Returns {group: {pairing: {...}}} with point estimates, bootstrap CIs and counts."""
    res = {}
    test = [r for r in rows if r["split"] == "test"]
    for g, dsets in groups.items():
        sub = [r for r in test if r["dataset"] in dsets]
        res[g] = {}
        for name in pairings:
            pos, neg = pairing(sub, name)
            if not pos and not neg:
                continue
            P, N = _by_question(pos, score_key), _by_question(neg, score_key)
            keys = sorted(set(P) | set(N))
            ps = [s for k in keys for s in P.get(k, [])]
            ns = [s for k in keys for s in N.get(k, [])]
            out = {"n_pos": len(ps), "n_neg": len(ns), "n_questions": len(keys)}
            point = {"auroc": auroc(ps, ns)}
            for tk, t in thresholds.items():
                point[f"tpr_{tk}"] = rate(ps, t); point[f"fpr_{tk}"] = rate(ns, t)
            rng = np.random.default_rng(seed); boots = {k: [] for k in point}
            for _ in range(n_boot):
                ks = _resample_keys(keys, rng)
                bp = [s for k in ks for s in P.get(k, [])]; bn = [s for k in ks for s in N.get(k, [])]
                boots["auroc"].append(auroc(bp, bn))
                for tk, t in thresholds.items():
                    boots[f"tpr_{tk}"].append(rate(bp, t)); boots[f"fpr_{tk}"].append(rate(bn, t))
            for k, v in point.items():
                b = np.asarray(boots[k], float); b = b[~np.isnan(b)]
                out[k] = {"value": None if np.isnan(v) else round(v, 4),
                          "ci95": [round(float(x), 4) for x in np.quantile(b, [0.025, 0.975])] if b.size else None}
            res[g][name] = out
    return res


def paired_auroc_difference(rows, key_a, key_b, name, group_dsets, n_boot=2000, seed=42):
    """Section 7: bootstrap the difference AUROC(a) - AUROC(b) on the same rows,
    resampling questions within dataset. Returns value, 95% CI and whether it excludes 0."""
    test = [r for r in rows if r["split"] == "test" and r["dataset"] in group_dsets]
    pos, neg = pairing(test, name)
    keys = sorted({(r["dataset"], r["qid"]) for r in pos + neg})
    Pa, Na, Pb, Nb = (_by_question(x, k) for x, k in ((pos, key_a), (neg, key_a), (pos, key_b), (neg, key_b)))
    def diff(ks):
        a = auroc([s for k in ks for s in Pa.get(k, [])], [s for k in ks for s in Na.get(k, [])])
        b = auroc([s for k in ks for s in Pb.get(k, [])], [s for k in ks for s in Nb.get(k, [])])
        return a - b
    point = diff(keys); rng = np.random.default_rng(seed)
    boots = np.array([diff(_resample_keys(keys, rng)) for _ in range(n_boot)])
    boots = boots[~np.isnan(boots)]
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return {"diff": round(float(point), 4), "ci95": [round(float(lo), 4), round(float(hi), 4)],
            "excludes_zero": bool(lo > 0 or hi < 0)}


def val_clean_n0(rows, score_key):
    """Section 7: thresholds are set on val n=0 clean passages (NQ + HotpotQA val)."""
    return [r[score_key] for r in rows
            if r["split"] == "val" and _is(r, group_variant="original", n_poison=0, label=0)]

In [ ]:
import importlib, numpy as np, pandas as pd, eval_core as ec
importlib.reload(ec)
ROWS = [json.loads(l) for l in open(f"{DRIVE_ROOT}/data/passages.jsonl")]
EVAL = [r for r in ROWS if r["split"] in ("val", "test") and r["context_id"] is not None]
KEY = {(r["context_id"], r["passage_id"]): r for r in EVAL}
print(f"{len(EVAL):,} val/test rows")

## 2. Attach every method's score to every row

In [ ]:
METHODS = {}          # name -> {"key": score column, "thresholds": {...}, "binary": bool}
_END = re.compile(r"[.!?][\"'\)\]]*$")
for r in EVAL:        # rule-based baselines computed from the text (protocol §8 definitions)
    w = r["passage"].split()
    r["starts_with_query"] = float(r["passage"].startswith(r["question"] + "."))
    r["ends_full_stop"] = float(bool(w) and bool(_END.search(w[-1])))
    r["len_score"] = -float(len(w))                         # shorter = poisoned (direction frozen on train)
METHODS["starts_with_query"] = {"key": "starts_with_query", "thresholds": {"rule": 0.5}, "binary": True}
METHODS["ends_full_stop"] = {"key": "ends_full_stop", "thresholds": {"rule": 0.5}, "binary": True}

# DistilBERT: 6 models, each with its own frozen thresholds
db = f"{DRIVE_ROOT}/step3_distilbert"
DB_MODELS = []
if os.path.exists(f"{db}/scores_distilbert.jsonl"):
    th = json.load(open(f"{db}/thresholds_distilbert.json"))["thresholds"]
    for l in open(f"{db}/scores_distilbert.jsonl"):
        d = json.loads(l); r = KEY[(d["context_id"], d["passage_id"])]
        for m in th: r[m] = d[m]
    for m in th:
        METHODS[m] = {"key": m, "thresholds": {k: v["threshold"] for k, v in th[m].items()}, "binary": False}
        DB_MODELS.append(m)
# Feature scorer, length, retrieval score and TrustRAG flags (Aleena's file and frozen thresholds)
fs = f"{DRIVE_ROOT}/step3_features/scores_features.jsonl"
if os.path.exists(fs):
    fth = json.load(open(f"{DRIVE_ROOT}/data/frozen/thresholds.json"))["thresholds"]
    for l in open(fs):
        d = json.loads(l); r = KEY.get((d["context_id"], d["passage_id"]))
        if r is None: continue
        for k in ("featscorer", "retrieval_score", "trustrag_removed"):
            if k in d: r[k] = float(d[k])
    def thr(name): return {k: v["threshold"] for k, v in fth[name].items()}
    METHODS["featscorer"] = {"key": "featscorer", "thresholds": thr("featscorer"), "binary": False}
    METHODS["length"] = {"key": "len_score", "thresholds": thr("length"), "binary": False}
    if all("retrieval_score" in r for r in EVAL):
        METHODS["retrieval_score"] = {"key": "retrieval_score", "thresholds": thr("retrieval"), "binary": False}
    if all("trustrag_removed" in r for r in EVAL):
        METHODS["trustrag"] = {"key": "trustrag_removed", "thresholds": {"rule": 0.5}, "binary": True}
missing = {m: sum(spec["key"] not in r for r in EVAL) for m, spec in METHODS.items()}
print("methods:", list(METHODS)); print("rows missing a score:", {k: v for k, v in missing.items() if v} or "none")
assert not any(missing.values()), "every method must score every val/test row"

## 3. Detection results (all pairings, both dataset groups)

In [ ]:
RES = {}
for m, spec in METHODS.items():
    nb = 0 if (m in DB_MODELS and not m.endswith("_s42")) else N_BOOT       # CIs for seed 42 and single models
    RES[m] = ec.evaluate(EVAL, spec["key"], spec["thresholds"], n_boot=nb, seed=SEED)
json.dump(RES, open(f"{OUT}/detection_results.json", "w"), indent=2)

def v(m, g, p, metric):
    x = RES[m].get(g, {}).get(p, {}).get(metric)
    return None if x is None else x["value"]
def ci(m, g, p, metric):
    x = RES[m].get(g, {}).get(p, {}).get(metric)
    return None if x is None or x["ci95"] is None else x["ci95"]

def fmt(m, g, p, metric):
    if METHODS.get(m, {}).get("binary") and metric == "auroc" and m == "trustrag":
        return "—"
    if m in ("aug", "noaug"):                                     # DistilBERT: mean ± sd over seeds, seed-42 CI
        vals = [v(f"{m}_s{s}", g, p, metric) for s in SEEDS]; vals = [x for x in vals if x is not None]
        if not vals: return "—"
        c = ci(f"{m}_s42", g, p, metric)
        return f"{np.mean(vals):.3f} ± {np.std(vals):.3f}" + (f" [{c[0]:.2f}, {c[1]:.2f}]" if c else "")
    x, c = v(m, g, p, metric), ci(m, g, p, metric)
    return "—" if x is None else f"{x:.3f}" + (f" [{c[0]:.2f}, {c[1]:.2f}]" if c else "")

LABEL = {"aug": "DistilBERT (augmented)", "noaug": "DistilBERT (no augmentation)", "featscorer": "Feature scorer",
         "length": "Length", "retrieval_score": "Retrieval score", "starts_with_query": "Starts with query",
         "ends_full_stop": "Ends with full stop", "trustrag": "TrustRAG Stage-1 (simplified)"}
ROWNAMES = [m for m in ["aug", "noaug", "featscorer", "retrieval_score", "length", "starts_with_query",
                        "ends_full_stop", "trustrag"] if m in METHODS or f"{m}_s42" in METHODS]
def mkey(m): return f"{m}_s42" if m in ("aug", "noaug") else m
cols = {}
for g in ec.GROUPS:
    for p in ("original", "shortcut_controlled"):
        cols[f"{g} | {p} | AUROC"] = {LABEL[m]: fmt(m, g, p, "auroc") for m in ROWNAMES}
        for m in ROWNAMES:
            k = "rule" if METHODS[mkey(m)]["binary"] else "fpr5"
            cols.setdefault(f"{g} | {p} | TPR (FPR) @5%", {})[LABEL[m]] = (
                f"{fmt(m, g, p, 'tpr_' + k)} ({fmt(m, g, p, 'fpr_' + k)})")
            cols.setdefault(f"{g} | {p} | TPR (FPR) @1%", {})[LABEL[m]] = (      # yes/no rules have one operating point
                "—" if METHODS[mkey(m)]["binary"] else f"{fmt(m, g, p, 'tpr_fpr1')} ({fmt(m, g, p, 'fpr_fpr1')})")
    for m in ROWNAMES:                     # false alarms on clean passages inside n=1 groups (protocol §7)
        k = "rule" if METHODS[mkey(m)]["binary"] else "fpr5"
        cols.setdefault(f"{g} | collateral FPR @5% (clean in n=1)", {})[LABEL[m]] = fmt(m, g, "collateral_n1", "fpr_" + k)
TABLE1 = pd.DataFrame(cols)
TABLE1.to_csv(f"{OUT}/table1_detection.csv")
display(TABLE1)

## 4. Every pairing, headline methods (for the Results text)

In [ ]:
PAIRS = ["original", "original_n5", "prefix_poison_side", "prefix_poison_side_n5", "prefix_clean_side", "llm_style", "shortcut_controlled",
         "unseen_attack", "collateral_n1"]
for g in ec.GROUPS:
    print(f"\n=== {g}")
    display(pd.DataFrame({LABEL[m]: {p: f"AUROC {fmt(m, g, p, 'auroc')} | TPR@5% {fmt(m, g, p, 'tpr_' + ('rule' if METHODS[mkey(m)]['binary'] else 'fpr5'))} | FPR@5% {fmt(m, g, p, 'fpr_' + ('rule' if METHODS[mkey(m)]['binary'] else 'fpr5'))}"
                                         for p in PAIRS} for m in ["aug", "noaug", "featscorer"] if mkey(m) in METHODS}))

## 5. Paired comparisons
**Pre-registered (protocol §7):** DistilBERT (augmented, each seed) − feature scorer, on the Original and Shortcut-controlled pairings, each dataset group.
**Additional (not pre-registered, label it so in the report):** augmented − not augmented, same seed, same pairings — the size of the augmentation effect.

In [ ]:
PAIRED = {"preregistered_distilbert_minus_featscorer": {}, "additional_aug_minus_noaug": {}}
for g, dsets in ec.GROUPS.items():
    for p in ("original", "shortcut_controlled"):
        for s in SEEDS:
            if f"aug_s{s}" in METHODS and "featscorer" in METHODS:
                PAIRED["preregistered_distilbert_minus_featscorer"][f"{g}|{p}|seed{s}"] = \
                    ec.paired_auroc_difference(EVAL, f"aug_s{s}", "featscorer", p, dsets, n_boot=N_BOOT, seed=SEED)
            if f"aug_s{s}" in METHODS and f"noaug_s{s}" in METHODS:
                PAIRED["additional_aug_minus_noaug"][f"{g}|{p}|seed{s}"] = \
                    ec.paired_auroc_difference(EVAL, f"aug_s{s}", f"noaug_s{s}", p, dsets, n_boot=N_BOOT, seed=SEED)
json.dump(PAIRED, open(f"{OUT}/paired_tests.json", "w"), indent=2)
for name, d in PAIRED.items():
    print(f"\n{name}")
    for k, x in d.items():
        print(f"  {k:45} ΔAUROC {x['diff']:+.3f}  95% CI [{x['ci95'][0]:+.3f}, {x['ci95'][1]:+.3f}]  excludes 0: {x['excludes_zero']}")

## 6. Figure 3 — shortcut tests

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
SERIES = [("aug_s42", "DistilBERT (augmented)", "#2a78d6", "o"),          # validated categorical slots 1-3
          ("noaug_s42", "DistilBERT (no aug.)", "#eb6834", "s"),
          ("featscorer", "Feature scorer", "#1baf7a", "^")]
SERIES = [s for s in SERIES if s[0] in METHODS]
XP = [("original", "Original"), ("prefix_poison_side", "Prefix\nremoved"), ("prefix_clean_side", "Clean\n+ prefix"),
      ("llm_style", "Rewritten\nclean"), ("shortcut_controlled", "Shortcut-\ncontrolled"), ("unseen_attack", "Injection\n(unseen)")]
plt.rcParams.update({"font.size": 8, "font.family": "serif"})
fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.6), dpi=300, sharey=True)
for ax, (g, title) in zip(axes, [("in_dist", "NQ + HotpotQA (test)"), ("msmarco", "MS MARCO (held out)")]):
    xs = [i for i, (p, _) in enumerate(XP) if any(v(m, g, p, "auroc") is not None for m, *_ in SERIES)]
    for j, (m, lab, col, mk) in enumerate(SERIES):
        dx = (j - (len(SERIES) - 1) / 2) * 0.18
        for i in xs:
            p = XP[i][0]; y = v(m, g, p, "auroc"); c = ci(m, g, p, "auroc")
            if y is None: continue
            ax.errorbar(i + dx, y, yerr=[[y - c[0]], [c[1] - y]] if c else None, fmt=mk, color=col, ms=5,
                        mec="white", mew=0.6, elinewidth=1.2, capsize=2, label=lab if i == xs[0] else None)
    ax.axhline(0.5, color="#8a8984", lw=0.8, ls="--"); ax.text(xs[0] - 0.45, 0.515, "chance", fontsize=6.5, color="#52514e", ha="left")
    ax.set_xticks(xs); ax.set_xticklabels([XP[i][1] for i in xs], fontsize=6.8)
    ax.set_title(title, fontsize=8.5); ax.set_ylim(0, 1.02); ax.grid(axis="y", color="#e4e3df", lw=0.6)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("AUROC (95% CI, seed 42)")
axes[0].legend(frameon=False, fontsize=7, loc="lower left")
fig.tight_layout()
fig.savefig(f"{OUT}/fig3_shortcuts.png"); fig.savefig(f"{OUT}/fig3_shortcuts.pdf")
plt.show()

## 7. End-to-end results, Figure 4 and the sensitivity check

In [ ]:
import glob
from collections import defaultdict
E2E = f"{DRIVE_ROOT}/step5_e2e"
def load_e2e(defence):
    recs = {}
    for p in sorted(glob.glob(f"{E2E}/records_{defence}_*.jsonl")):
        for d in map(json.loads, open(p)): recs[(d["dataset"], d["qid"], d["n_poison"])] = d
    return list(recs.values())
summary = json.load(open(f"{E2E}/summary.json")) if os.path.exists(f"{E2E}/summary.json") else {}
DEF = [("none", "No defence"), ("distilbert", "DistilBERT"), ("featscorer", "Feature scorer"), ("trustrag", "TrustRAG S1")]
DEF = [d for d in DEF if any(k.split("|")[1] == d[0] for k in summary)]
if DEF:
    fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.3), dpi=300)
    panels = [("n=1|ASR (substring)", "Attack success, n = 1"), ("n=5|ASR (substring)", "Attack success, n = 5"),
              ("n=0|F1 vs correct", "Clean-answer F1, n = 0")]
    for ax, (key, title) in zip(axes, panels):
        for j, (g, lab, col, mk) in enumerate([("in_dist", "NQ + HotpotQA", "#2a78d6", "o"), ("msmarco", "MS MARCO", "#eb6834", "s")]):
            for i, (d, _) in enumerate(DEF):
                n, metric = key.split("|")
                c = summary.get(f"{g}|{d}|{n}|{metric}")
                if not c: continue
                ax.errorbar(i + (j - 0.5) * 0.25, c[0], yerr=[[c[0] - c[1]], [c[2] - c[0]]], fmt=mk, color=col, ms=5,
                            mec="white", mew=0.6, elinewidth=1.2, capsize=2, label=lab if i == 0 else None)
        ax.set_xticks(range(len(DEF))); ax.set_xticklabels([x[1] for x in DEF], fontsize=6.5, rotation=20, ha="right")
        ax.set_title(title, fontsize=8.5); ax.set_ylim(0, 1.02); ax.grid(axis="y", color="#e4e3df", lw=0.6)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
    axes[0].legend(frameon=False, fontsize=7, loc="upper right")
    fig.tight_layout(); fig.savefig(f"{OUT}/fig4_end_to_end.png"); fig.savefig(f"{OUT}/fig4_end_to_end.pdf"); plt.show()

# Sensitivity check (protocol §10): n=1 ASR with hand-check-flagged test questions excluded
SENS = {}
for d, _ in DEF:
    recs = [r for r in (load_e2e(d) if d != "none" else []) if r["n_poison"] == 1]
    if d == "none":
        recs = []
        for ds, folder in {"nq": "step1", "hotpotqa": "step1_hotpotqa", "msmarco": "step1_msmarco"}.items():
            test = set(json.load(open(f"{DRIVE_ROOT}/{folder}/split.json"))["test"])
            for r in map(json.loads, open(f"{DRIVE_ROOT}/{folder}/records.jsonl")):
                if r["n_poison"] == 1 and r["qid"] in test:
                    recs.append({"dataset": ds, "qid": r["qid"], "says_incorrect": r["says_incorrect"]})
    for g, dsets in ec.GROUPS.items():
        allr = [r for r in recs if r["dataset"] in dsets]
        kept = [r for r in allr if (r["dataset"], r["qid"]) not in FLAGGED]
        if allr:
            SENS[f"{g}|{d}"] = {"asr_all": round(float(np.mean([r["says_incorrect"] for r in allr])), 3), "n_all": len(allr),
                                "asr_excl_flagged": round(float(np.mean([r["says_incorrect"] for r in kept])), 3), "n_kept": len(kept)}
json.dump(SENS, open(f"{OUT}/sensitivity.json", "w"), indent=2)
display(pd.DataFrame(SENS).T)

## 7b. Table 2: end-to-end, with every protocol §9 metric, and latency
Adds what the run notebook's preview leaves out: attack success **given that poison was retrieved**, the clean-passage **removal rate** (removed ÷ clean passages in the group) next to the mean count per question, and **latency per passage**. The feature scorer's n=5 rows are outside its training distribution (protocol §5) and are marked so.

In [ ]:
%%writefile rag_core.py
"""Core, model-free logic for the Step 1 baseline.

Everything here is pure Python / NumPy so it can be unit-tested without a GPU.
The notebook supplies the retriever scores and the generator responses.
"""
import re
import string
from collections import Counter

import numpy as np


# ---------------------------------------------------------------- answer matching

def poisonedrag_clean_str(s):
    """Exact replica of PoisonedRAG's clean_str (src/utils.py).

    Used for the primary attack-success criterion so our ASR is comparable
    with the numbers reported by Zou et al.
    """
    s = str(s).strip()
    if len(s) > 1 and s[-1] == ".":
        s = s[:-1]
    return s.lower()


def contains_answer(response, answer):
    """PoisonedRAG's success test: the cleaned answer is a substring of the cleaned response."""
    return poisonedrag_clean_str(answer) in poisonedrag_clean_str(response)


def normalize_answer(s):
    """SQuAD-style normalisation: lowercase, strip punctuation, articles and extra whitespace."""
    s = str(s).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def token_f1(prediction, gold):
    """SQuAD token-level F1 between a free-form prediction and a gold answer."""
    pred_toks = normalize_answer(prediction).split()
    gold_toks = normalize_answer(gold).split()
    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks)
    common = Counter(pred_toks) & Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)


def contains_answer_word(response, answer):
    """Whole-word variant of contains_answer, on SQuAD-normalised text.

    Fixes false matches of PoisonedRAG's substring test such as "no" inside
    "I don't know", "Norway" or "Rohypnol". It still misses paraphrased yes/no
    answers ("X is not Y" for "no"), so the two tests bracket the true rate:
    word match is a lower bound, substring an upper bound.
    """
    import re as _re
    a = normalize_answer(answer)
    r = normalize_answer(response)
    return bool(a) and _re.search(r"(?<!\w)" + _re.escape(a) + r"(?!\w)", r) is not None


# ---------------------------------------------------------------- poisoned passages

def build_poisoned_passages(question, adv_texts, n):
    """PoisonedRAG black-box ('LM_targeted') construction, src/attack.py:
        adv_text_a = question + "."
        adv_texts  = [adv_text_a + b for b in adv_texts_b]
    The question prefix (S) gets the passage retrieved; the released text (I)
    carries the false answer. The released JSON contains only I.
    """
    return [question + "." + t for t in adv_texts[:n]]


# ---------------------------------------------------------------- retrieval with injection

def merge_topk(clean_hits, adv_hits, k):
    """Top-k of (clean corpus ∪ injected passages).

    clean_hits: list of (doc_id, score) — the clean retriever's ranking over the
                full corpus, best first (at least k entries).
    adv_hits:   list of (adv_id, score) for the injected passages.

    Because retrieval is exact maximum-inner-product search, the top-k of the
    union equals the top-k of (clean top-k ∪ injected), so injecting into the
    full 2.68M-passage NQ corpus does not require re-indexing it.
    Ties keep clean passages first (conservative towards the attack).
    """
    if len(clean_hits) < k:
        raise ValueError(f"need at least k={k} clean hits, got {len(clean_hits)}")
    pool = [{"source": "clean", "id": d, "score": float(s)} for d, s in clean_hits[:k]]
    pool += [{"source": "adv", "id": a, "score": float(s)} for a, s in adv_hits]
    # stable sort: clean entries were added first, so they win ties
    pool.sort(key=lambda r: -r["score"])
    return pool[:k]


# ---------------------------------------------------------------- statistics

def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=0):
    """Mean and percentile bootstrap CI over questions (the unit of resampling)."""
    v = np.asarray(values, dtype=float)
    if v.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, v.size, size=(n_boot, v.size))
    means = v[idx].mean(axis=1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return float(v.mean()), float(lo), float(hi)


def summarise(records, n_boot=2000, seed=0):
    """Aggregate per-question records for one condition into the Step 1 metrics."""
    out = {"n_questions": len(records)}
    metrics = {
        "retrieval_success": [r["n_adv_in_topk"] > 0 for r in records],
        "adv_in_topk_mean": [r["n_adv_in_topk"] for r in records],
        "asr": [r["says_incorrect"] for r in records],
        "clean_acc_contains": [r["says_correct"] for r in records],
        "clean_f1": [r["f1_correct"] for r in records],
        "both_answers": [r["says_incorrect"] and r["says_correct"] for r in records],
    }
    # Conditional ASR: among questions where a poisoned passage reached the top-k,
    # how often did the generator adopt the attacker's answer? Separates "not
    # retrieved" from "retrieved but not believed".
    metrics["asr_given_retrieved"] = [r["says_incorrect"] for r in records if r["n_adv_in_topk"] > 0]
    for name, vals in metrics.items():
        m, lo, hi = bootstrap_ci(vals, n_boot=n_boot, seed=seed)
        out[name] = {"mean": round(m, 4), "ci95": [round(lo, 4), round(hi, 4)], "n": len(vals)}
    return out


# ---------------------------------------------------------------- data split

def split_questions(qids, fractions=(0.6, 0.2, 0.2), seed=42):
    """Fixed train/val/test split by target question, decided before any detector work."""
    if abs(sum(fractions) - 1) > 1e-9:
        raise ValueError("fractions must sum to 1")
    qids = sorted(qids)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(qids))
    n_train = int(round(fractions[0] * len(qids)))
    n_val = int(round(fractions[1] * len(qids)))
    shuffled = [qids[i] for i in order]
    return {
        "train": sorted(shuffled[:n_train]),
        "val": sorted(shuffled[n_train:n_train + n_val]),
        "test": sorted(shuffled[n_train + n_val:]),
    }

In [ ]:
import importlib, rag_core as rc
importlib.reload(rc)
STEP1_DIR = {"nq": f"{DRIVE_ROOT}/step1", "hotpotqa": f"{DRIVE_ROOT}/step1_hotpotqa", "msmarco": f"{DRIVE_ROOT}/step1_msmarco"}
TEST_Q = {(r["dataset"], r["qid"]) for r in ROWS if r["split"] == "test"}
UND, GEN_S = [], []
for ds, d in STEP1_DIR.items():
    for r in map(json.loads, open(f"{d}/records.jsonl")):
        if r["n_poison"] in (0, 1, 5) and (ds, r["qid"]) in TEST_Q:
            UND.append({"dataset": ds, "qid": r["qid"], "n_poison": r["n_poison"], "n_adv_in_topk": r["n_adv_in_topk"],
                        "n_adv_kept": r["n_adv_in_topk"], "n_clean_removed": 0, "all_removed": False,
                        "says_incorrect": rc.contains_answer(r["response"], r["incorrect"]),
                        "says_incorrect_word": rc.contains_answer_word(r["response"], r["incorrect"]),
                        "says_correct": rc.contains_answer(r["response"], r["correct"]),
                        "f1_correct": rc.token_f1(r["response"], r["correct"])})
            if "gen_seconds" in r: GEN_S.append(r["gen_seconds"])
REC = {"none": UND, **{d: load_e2e(d) for d in ("distilbert", "featscorer", "trustrag")}}
REC = {d: v for d, v in REC.items() if v}

def boot_mean(recs, fn, dsets):
    by = {}
    for r in recs:
        if r["dataset"] in dsets:
            x = fn(r)
            if x is not None and not (isinstance(x, float) and np.isnan(x)): by.setdefault(r["dataset"], []).append(float(x))
    if not by: return None
    allv = np.concatenate([np.asarray(v) for v in by.values()])
    rng = np.random.default_rng(SEED)
    b = [np.concatenate([np.asarray(v)[rng.integers(0, len(v), len(v))] for v in by.values()]).mean() for _ in range(N_BOOT)]
    lo, hi = np.quantile(b, [0.025, 0.975])
    return f"{allv.mean():.3f} [{lo:.3f}, {hi:.3f}]", int(allv.size)

M2 = {"ASR (substring)": lambda r: r["says_incorrect"],
      "ASR (whole word)": lambda r: r["says_incorrect_word"],
      "ASR | poison retrieved": lambda r: r["says_incorrect"] if r["n_adv_in_topk"] > 0 else None,
      "clean removal rate": lambda r: r["n_clean_removed"] / (5 - r["n_adv_in_topk"]) if r["n_adv_in_topk"] < 5 else None,
      "clean removed per question (mean count)": lambda r: r["n_clean_removed"],
      "all 5 removed": lambda r: r["all_removed"],
      "F1 vs correct": lambda r: r["f1_correct"]}
T2 = []
for g, dsets in ec.GROUPS.items():
    base_f1 = np.mean([r["f1_correct"] for r in REC["none"] if r["n_poison"] == 0 and r["dataset"] in dsets])
    for d, recs in REC.items():
        for n in (0, 1, 5):
            sub = [r for r in recs if r["n_poison"] == n and r["dataset"] in dsets]
            if not sub: continue
            row = {"group": g, "defence": d, "n": n, "questions": len(sub)}
            for name, fn in M2.items():
                res = boot_mean(sub, fn, dsets); row[name] = res[0] if res else "—"
            row["F1 retention (n=0)"] = (f"{np.mean([r['f1_correct'] for r in sub]) / base_f1:.3f}" if n == 0 and d != "none" else "—")
            row["note"] = "outside training distribution (protocol §5)" if (d == "featscorer" and n == 5) else ""
            T2.append(row)
TABLE2 = pd.DataFrame(T2).set_index(["group", "defence", "n"])
TABLE2.to_csv(f"{OUT}/table2_end_to_end.csv"); display(TABLE2)

# Latency per passage (protocol §9)
LAT = {}
tl = json.load(open(f"{db}/train_log.json")) if os.path.exists(f"{db}/train_log.json") else {}
ms = [v["score_ms_per_passage"] for v in tl.values() if isinstance(v, dict) and "score_ms_per_passage" in v]
if ms: LAT["DistilBERT, ms per passage (T4, batched; median of models)"] = round(float(np.median(ms)), 2)
lp = f"{DRIVE_ROOT}/step3_features/latency.json"           # Aleena: feature scorer and TrustRAG timings
if os.path.exists(lp): LAT.update(json.load(open(lp)))
else: print("step3_features/latency.json not found: feature-scorer and TrustRAG latency missing")
if GEN_S: LAT["Generator Qwen2.5-3B, s per answer (T4, median, for scale)"] = round(float(np.median(GEN_S)), 2)
json.dump(LAT, open(f"{OUT}/latency.json", "w"), indent=2); print(json.dumps(LAT, indent=2))

## 8. Failure-analysis sheets (20 misses + 20 false alarms per detector, seed 42)

In [ ]:
import csv
rng = random.Random(SEED)
test = [r for r in EVAL if r["split"] == "test"]
for m in [x for x in ("aug_s42", "featscorer") if x in METHODS]:
    t = METHODS[m]["thresholds"]["fpr5"]
    pos, neg = ec.pairing(test, "original")
    spos, sneg = ec.pairing(test, "shortcut_controlled")
    misses = [r for r in pos + spos if r[m] <= t]
    alarms = [r for r in neg + sneg if r[m] > t]
    pick = rng.sample(misses, min(20, len(misses))) + rng.sample(alarms, min(20, len(alarms)))
    with open(f"{OUT}/failure_{m}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["type", "dataset", "qid", "group_variant", "variant", "score", "threshold", "question", "passage",
                    "category", "note"])
        for r in pick:
            w.writerow(["miss" if r["label"] == 1 else "false_alarm", r["dataset"], r["qid"], r["group_variant"], r["variant"],
                        round(r[m], 4), round(t, 4), r["question"], r["passage"], "", ""])
    print(f"{m}: {len(misses)} misses, {len(alarms)} false alarms in total -> failure_{m}.csv ({len(pick)} sampled)")